# Image Feature Extraction — CLIP ViT-B/32

用 CLIP visual encoder 提取 43,528 item 图片特征 (512-dim)。
缺失图片填充零向量。

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch, json, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel

META_CSV    = "final/item_meta_merged.csv"
IMAGE_MAP   = "final/item_image_map.json"
IMAGE_DIR   = "final/images"
OUTPUT_DIR  = "final/features"
CLIP_MODEL  = "openai/clip-vit-base-patch32"
BATCH_SIZE  = 64
FEATURE_DIM = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [3]:
model = CLIPModel.from_pretrained(CLIP_MODEL).to(device).eval()
processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
torch.set_grad_enabled(False)
print(f"Loaded: {CLIP_MODEL}")

Loaded: openai/clip-vit-base-patch32


In [ ]:
meta = pd.read_csv(META_CSV)
print(f"Items: {len(meta)}")

with open(IMAGE_MAP) as f:
    img_map = json.load(f)

img_paths = {}
missing = 0
for _, row in meta.iterrows():
    iid = int(row["parent_asin"])
    key = str(iid)
    if key in img_map:
        path = img_map[key].replace("\\", "/")
        if not path.startswith("final/"):
            path = f"final/images/{iid}.jpg"
    else:
        path = None
        missing += 1
    img_paths[iid] = path

print(f"Mapped: {len(img_paths)-missing}  Missing: {missing}")

Items: 43528
Mapped: 43509  Missing: 19


In [5]:
class ImageDataset(Dataset):
    def __init__(self, item_ids, path_dict):
        self.ids = list(item_ids)
        self.paths = [path_dict[i] for i in self.ids]
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, idx):
        return self.ids[idx], self.paths[idx]

def collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]

item_ids = sorted(img_paths.keys())
loader = DataLoader(ImageDataset(item_ids, img_paths),
                    batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=collate, num_workers=0)

### 特征提取

In [6]:
all_feats, all_ids, fail = [], [], 0

for ids_batch, paths_batch in tqdm(loader, desc="Image"):
    images, valid_ix = [], []
    for i, p in enumerate(paths_batch):
        if p and os.path.exists(p):
            try:
                images.append(Image.open(p).convert("RGB"))
                valid_ix.append(i)
            except Exception:
                pass

    feats = np.zeros((len(ids_batch), FEATURE_DIM), dtype=np.float32)
    if images:
        inp = processor(images=images, return_tensors="pt").to(device)
        with torch.no_grad():
            vf = model.get_image_features(**inp).cpu().numpy()
        for j, f in zip(valid_ix, vf):
            feats[j] = f
    else:
        fail += len(ids_batch)

    all_feats.append(feats)
    all_ids.extend(ids_batch)

all_feats = np.concatenate(all_feats, axis=0)
print(f"Done.  shape={all_feats.shape}  zeros={fail}")

Image: 100%|██████████| 681/681 [31:59<00:00,  2.82s/it]

Done.  shape=(43528, 512)  zeros=0


In [7]:
# ==================== Save ====================
np.save(f"{OUTPUT_DIR}/image_features_512.npy", all_feats)
pd.DataFrame({"item_id": all_ids}).to_csv(f"{OUTPUT_DIR}/image_id_map.csv", index=False)
print(f"Saved to {OUTPUT_DIR}/image_features_512.npy")
print(f"Zero rows: {(all_feats.sum(1)==0).sum()} (expect ~19)")

Saved to final/features/image_features_512.npy
Zero rows: 19 (expect ~19)
